# Unwrapping AI | Zero to Transformer
## Step 09: Random Forests — Beginner's Learning Book
**Creator:** Nitin K. Sethi

### Learning objectives
- Explain ensemble learning, bootstrap sampling and random feature selection.
- Train a Random Forest classifier and compare it with one Decision Tree.
- Evaluate classification results.
- Tune hyperparameters with cross-validation.
- Inspect feature importance and its limitations.

> Educational exercise using Scikit-learn's Breast Cancer Wisconsin dataset. This is not a medical decision tool.

## 1. What is a Random Forest?

A Random Forest combines predictions from many Decision Trees. It introduces diversity through **bootstrap sampling** (random training rows sampled with replacement) and **random feature selection** (only a subset of features is considered at each split). For classification, trees vote and the forest predicts the majority class.

Combining diverse trees can reduce the variance of a single tree, although a Random Forest is not guaranteed to outperform every other model.

## 2. Key terms
- `n_estimators`: number of trees.
- `max_features`: features considered at each split.
- `max_depth`: maximum depth of each tree.
- `min_samples_leaf`: minimum samples required at a leaf.
- **Out-of-bag (OOB) score:** a training-based estimate using rows left out of an individual tree's bootstrap sample; available when `bootstrap=True`.

## 3. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

## 4. Load and inspect the dataset
We reuse the Step 08 dataset for a fair comparison.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X, y = dataset.data, dataset.target
print("Feature matrix:", X.shape)
print("Target mapping:", dict(enumerate(dataset.target_names)))
print("Class counts:\\n", y.value_counts())
print("Missing values:", int(X.isna().sum().sum()))
display(X.head())

## 5. Create a stratified train-test split
Keep the test set untouched during model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Compare a Decision Tree with a baseline Random Forest

In [ ]:
single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
forest = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE,
    n_jobs=-1, oob_score=True
)
single_tree.fit(X_train, y_train)
forest.fit(X_train, y_train)

comparison = pd.DataFrame([
    {"model": "Decision Tree",
     "train_accuracy": accuracy_score(y_train, single_tree.predict(X_train)),
     "test_accuracy": accuracy_score(y_test, single_tree.predict(X_test))},
    {"model": "Random Forest",
     "train_accuracy": accuracy_score(y_train, forest.predict(X_train)),
     "test_accuracy": accuracy_score(y_test, forest.predict(X_test))}
])
display(comparison)
print("Random Forest OOB score:", round(forest.oob_score_, 3))

### Think about it
Which model has a larger gap between training and test accuracy? Why might combining many trees make predictions more stable? Performance depends on the dataset and settings.

## 7. Evaluate the baseline Random Forest

In [ ]:
baseline_pred = forest.predict(X_test)
print(classification_report(y_test, baseline_pred, target_names=dataset.target_names))
ConfusionMatrixDisplay.from_predictions(
    y_test, baseline_pred, display_labels=dataset.target_names,
    values_format="d", cmap="Blues"
)
plt.title("Baseline Random Forest — test confusion matrix")
plt.show()

## 8. Observe forest size (illustration only)

This section is exploratory. Do not choose hyperparameters by repeatedly checking test-set results; use cross-validation on the training data in the next section.

In [ ]:
tree_counts = [10, 50, 100, 200, 300]
rows = []
for count in tree_counts:
    model = RandomForestClassifier(
        n_estimators=count, random_state=RANDOM_STATE, n_jobs=-1
    ).fit(X_train, y_train)
    rows.append({
        "n_estimators": count,
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, model.predict(X_test))
    })
size_results = pd.DataFrame(rows)
display(size_results)
size_results.plot(x="n_estimators", y=["train_accuracy", "test_accuracy"],
                  marker="o", figsize=(8, 4), ylim=(0.85, 1.01))
plt.title("Observed accuracy by forest size")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

## 9. Hyperparameter tuning with cross-validation
GridSearchCV uses only the training partition. The test set is not passed into the search.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", 0.5],
    "min_samples_leaf": [1, 2, 4]
}
search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid, scoring="f1_macro",
    cv=cv, n_jobs=-1, refit=True
)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best mean CV macro F1:", round(search.best_score_, 3))

## 10. Final evaluation on the untouched test set

In [ ]:
best_forest = search.best_estimator_
final_pred = best_forest.predict(X_test)
print("Final test accuracy:", round(accuracy_score(y_test, final_pred), 3))
print(classification_report(y_test, final_pred, target_names=dataset.target_names))
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, display_labels=dataset.target_names,
    values_format="d", cmap="Blues"
)
plt.title("Tuned Random Forest — final test results")
plt.show()

## 11. Feature importance

Impurity-based importance describes each feature's contribution to impurity reduction in the fitted forest. Permutation importance measures the change in a chosen score when a feature is shuffled.

Neither establishes causation. Correlated features can share or mask importance, and permutation importance depends on the evaluation data and scoring metric.

In [ ]:
imp = pd.Series(best_forest.feature_importances_, index=X.columns).sort_values(ascending=False)
display(imp.head(10).to_frame("impurity_importance"))

perm = permutation_importance(
    best_forest, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=-1
)
perm_scores = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(perm_scores.head(10).to_frame("mean_permutation_importance"))
perm_scores.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top permutation importances on test data")
plt.xlabel("Mean decrease in macro F1 after shuffling")
plt.show()

## 12. Practical exercises
1. Try `n_estimators` values of 100, 200 and 400; compare training time and CV score.
2. Compare `max_features="sqrt"` with `max_features=0.5`.
3. Increase `min_samples_leaf` and observe model complexity.
4. Compare the Decision Tree and tuned Random Forest using macro F1.
5. Explain bootstrap sampling in your own words.
6. Why is the test set reserved for final evaluation?

### Reflection
Write 3–5 sentences explaining why a Random Forest can be more stable than one Decision Tree and identify one trade-off.

## 13. Key takeaways
- A Random Forest combines many Decision Trees.
- Bootstrap sampling and random feature selection create diversity.
- Classification predictions are combined through voting.
- OOB score is an additional estimate when bootstrap sampling is enabled.
- Cross-validation selects hyperparameters using training data.
- The held-out test set is reserved for final evaluation.
- Feature importance must be interpreted carefully.

**Next milestone:** Step 10 — Perceptron.

## References
- [Scikit-learn Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [RandomForestClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [GridSearchCV API](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)